In [2]:
# ============================================================
# Imports y configuración global del entorno (versión v4-1001)
# ============================================================
# Esta celda inicializa el entorno para un nuevo experimento
# donde el TARGET_LENGTH cambia de 3197 → 1001.
#
# ¿Por qué es importante este cambio?
#   ✔ Reduce el tamaño de entrada a la SNN
#   ✔ Acelera entrenamiento e inferencia
#   ✔ Permite experimentar con resoluciones más compactas
#   ✔ Útil para modelos más ligeros o despliegue en producción
#
# El resto del pipeline (descarga, preprocesamiento, alineación,
# entrenamiento, evaluación) se adapta automáticamente a este valor.
# ============================================================

import os, time, warnings, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PyTorch + SNNtorch (modelo bioinspirado)
# ------------------------------------------------------------
import torch
import torch.nn as nn
import snntorch as snn
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# ------------------------------------------------------------
# Utilidades de Machine Learning
# ------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score
)

# ------------------------------------------------------------
# Lightkurve: descarga y manejo de curvas de luz Kepler
# ------------------------------------------------------------
import lightkurve as lk

# Silenciar warnings comunes de Lightkurve
warnings.filterwarnings("ignore", category=UserWarning, module="lightkurve")

# ------------------------------------------------------------
# Reproducibilidad global
# ------------------------------------------------------------
torch.manual_seed(42)
np.random.seed(42)

# ------------------------------------------------------------
# Parámetros globales del experimento v4-1001
# ------------------------------------------------------------
TARGET_LENGTH = 1001          # <-- CAMBIO CLAVE: nueva longitud de entrada
BATCH_SIZE    = 32            # Tamaño de batch
NUM_STEPS_SNN = 25            # Pasos temporales de la SNN bioinspirada
DEVICE        = torch.device('cpu')  # CPU para reproducibilidad

# ------------------------------------------------------------
# Directorios del proyecto
# ------------------------------------------------------------
DATA_DIR    = '../data'
MODELS_DIR  = '../models'
OUTPUTS_DIR = '../outputs'

# Crear directorios si no existen
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print(f"TARGET_LENGTH = {TARGET_LENGTH}")
print("Imports OK")


TARGET_LENGTH = 1001
Imports OK


d:\AI\Conectoma\exoplanetas-snn\.venv\Lib\site-packages\lightkurve\prf\__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [3]:
# ============================================================
# Funciones auxiliares para el pipeline Kepler v4‑1001
# ============================================================
# Estas funciones implementan el flujo completo para preparar
# curvas de luz reales antes de alimentar el modelo SNN:
#
#   1. get_stitched_lightcurve → descarga + unión de quarters
#   2. preprocess_lightcurve_own_v2 → limpieza preservando tránsitos
#   3. align_to_length → interpolación a TARGET_LENGTH (1001)
#
# Son idénticas en comportamiento a las usadas en v3/v4‑3197,
# excepto que ahora la alineación se hace a un tamaño variable
# definido por el usuario (target_length).
# ============================================================

def get_stitched_lightcurve(kepid, cadence='long', author='Kepler'):
    """
    Descarga y une todos los trimestres de la curva de luz de una estrella Kepler.

    Flujo:
      1. Buscar todos los segmentos disponibles para el KEPID.
      2. Descargar todos los quarters.
      3. Unirlos en una sola curva continua (stitch).

    Retorna
    -------
    LightCurve o None
        Curva completa ya unificada. None si falla la descarga.

    Notas:
    -------
    - Es la función estándar para obtener curvas reales.
    - Kepler observa cada estrella en múltiples quarters → stitch es obligatorio.
    """
    try:
        search_result = lk.search_lightcurve(f"KIC {kepid}", author=author, cadence=cadence)
        if search_result is None or len(search_result) == 0:
            return None

        lc_collection = search_result.download_all()
        if lc_collection is None or len(lc_collection) == 0:
            return None

        return lc_collection.stitch()

    except Exception as e:
        print(f"  [WARN] {type(e).__name__}: {e}")
        return None


def preprocess_lightcurve_own_v2(lc, sigma_upper=5):
    """
    Preprocesamiento que PRESERVA los tránsitos planetarios.

    Flujo:
      1. Quitar NaNs
      2. Normalizar por mediana (centra en ~1.0)
      3. Eliminar SOLO outliers positivos (cosmic rays, momentum dumps)

    Razón:
      - Los outliers positivos son instrumentales.
      - Los valores negativos suelen ser tránsitos → NO deben eliminarse.

    Retorna
    -------
    np.ndarray o None
        Curva preprocesada sin aplanado, preservando tránsitos.
    """
    try:
        lc = lc.remove_nans()
        flux = lc.flux.value.astype(np.float64)
        flux = flux[np.isfinite(flux)]

        if len(flux) < 100:
            return None

        median = np.median(flux)
        if median == 0 or not np.isfinite(median):
            return None

        flux = flux / median

        # Sigma-clipping solo para outliers positivos
        for _ in range(3):
            mean = np.mean(flux)
            std = np.std(flux)
            if std == 0:
                break

            mask = flux < (mean + sigma_upper * std)
            if mask.sum() < 100:
                break

            flux = flux[mask]

        return flux

    except Exception as e:
        print(f"  [WARN] {type(e).__name__}: {e}")
        return None


def align_to_length(flux, target_length):
    """
    Interpola la curva a una longitud fija (target_length).

    Flujo:
      1. Convertir a float32
      2. Quitar NaNs residuales
      3. Interpolar linealmente a longitud fija

    Razón:
      - La SNN requiere vectores de entrada de tamaño fijo.
      - La interpolación lineal preserva la forma general del tránsito.
      - Ahora target_length puede ser 3197, 1001 o cualquier valor.

    Retorna
    -------
    np.ndarray o None
        Vector alineado de longitud target_length.
    """
    flux = np.asarray(flux, dtype=np.float32)
    flux = flux[np.isfinite(flux)]

    if len(flux) < 100:
        return None

    x_old = np.linspace(0, 1, len(flux))
    x_new = np.linspace(0, 1, target_length)

    return np.interp(x_new, x_old, flux).astype(np.float32)


print("Funciones auxiliares OK")


Funciones auxiliares OK


In [4]:
checks = {
    'get_stitched_lightcurve':    'get_stitched_lightcurve' in dir(),
    'preprocess_lightcurve_own_v2': 'preprocess_lightcurve_own_v2' in dir(),
    'align_to_length':            'align_to_length' in dir(),
}
for name, ok in checks.items():
    print(f"  {name}: {'OK' if ok else 'FALTA'}")

  get_stitched_lightcurve: OK
  preprocess_lightcurve_own_v2: OK
  align_to_length: OK


In [1]:
import os
import numpy as np

DATA_DIR = '../data'
X_PATH = f'{DATA_DIR}/v5_X.npy'
y_PATH = f'{DATA_DIR}/v5_y.npy'
k_PATH = f'{DATA_DIR}/v5_kepids.npy'

if os.path.exists(X_PATH):
    X = np.load(X_PATH)
    y = np.load(y_PATH)
    k = np.load(k_PATH)
    print(f"Checkpoint existe")
    print(f"  Estrellas guardadas: {X.shape[0]}")
    print(f"  Distribucion: {np.bincount(y)}")
    print(f"  Kepids unicos: {len(set(k.tolist()))}")
else:
    print("No existe checkpoint. Se perdio todo el progreso.")

Checkpoint existe
  Estrellas guardadas: 2000
  Distribucion: [ 600 1400]
  Kepids unicos: 2000


In [5]:
# ============================================================
# DESCARGADOR INCREMENTAL DE DATASET KEPLER (v5)
# ============================================================
# Objetivo:
#   Construir un dataset grande y reproducible de curvas Kepler,
#   combinando:
#       ✔ N_PER_CLASS estrellas CONFIRMED
#       ✔ N_PER_CLASS estrellas Sin KOI
#
# Características clave:
#   ✔ Descarga incremental (resume automático)
#   ✔ Checkpoints periódicos
#   ✔ Preprocesamiento robusto preservando tránsitos
#   ✔ Alineación a TARGET_LENGTH_DS (1001 por defecto)
#   ✔ Reproducible (semilla fija)
#
# Este script permite construir datasets grandes (v5) sin perder
# progreso si la ejecución se interrumpe.
# ============================================================

import os
import time
import numpy as np
import pandas as pd

from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

# ---------- PARAMETROS ----------
N_PER_CLASS       = 1400      # <-- CAMBIA ESTO para ajustar tamaño del dataset
TARGET_LENGTH_DS  = 1001     # Longitud final de cada curva
CHECKPOINT_EVERY  = 50       # Guardar cada N estrellas
OUTPUT_PREFIX     = 'v5'     # Prefijo de archivos .npy
RANDOM_SEED       = 42       # Reproducibilidad
# --------------------------------

DATA_DIR = '../data'
X_PATH     = f'{DATA_DIR}/{OUTPUT_PREFIX}_X.npy'
y_PATH     = f'{DATA_DIR}/{OUTPUT_PREFIX}_y.npy'
kepid_PATH = f'{DATA_DIR}/{OUTPUT_PREFIX}_kepids.npy'


# ============================================================
# 1. Obtener listas de kepids (CONFIRMED + Sin KOI)
# ============================================================
print("Obteniendo listas de kepids...")

# ------------------------------------------------------------
# CONFIRMED desde el catálogo KOI completo
# ------------------------------------------------------------
koi_table = NasaExoplanetArchive.query_criteria(
    table="cumulative",
    select="kepid, koi_disposition"
)
koi_df_full = koi_table.to_pandas()

confirmed_kepids = (
    koi_df_full[koi_df_full['koi_disposition'] == 'CONFIRMED']['kepid']
    .dropna()
    .unique()
    .tolist()
)
confirmed_kepids = [int(k) for k in confirmed_kepids]
print(f"CONFIRMED disponibles: {len(confirmed_kepids)}")

# ------------------------------------------------------------
# Sin KOI desde archivo precomputado
# ------------------------------------------------------------
sin_koi_PATH = f'{DATA_DIR}/sin_koi_kepids.npy'
if not os.path.exists(sin_koi_PATH):
    raise FileNotFoundError(
        f"Falta {sin_koi_PATH}. Ejecuta la celda previa que genera sin_koi_kepids.npy"
    )

sin_koi_kepids_all = np.load(sin_koi_PATH).tolist()
sin_koi_kepids_all = [
    int(k) for k in sin_koi_kepids_all
    if int(k) not in set(confirmed_kepids)
]
print(f"Sin KOI disponibles: {len(sin_koi_kepids_all)}")


# ============================================================
# 2. Muestreo reproducible
# ============================================================
np.random.seed(RANDOM_SEED)

confirmed_sample = np.random.choice(
    confirmed_kepids,
    size=min(N_PER_CLASS, len(confirmed_kepids)),
    replace=False
).tolist()

sin_koi_sample = np.random.choice(
    sin_koi_kepids_all,
    size=min(N_PER_CLASS, len(sin_koi_kepids_all)),
    replace=False
).tolist()

target_kepids = confirmed_sample + sin_koi_sample
target_labels = [1] * len(confirmed_sample) + [0] * len(sin_koi_sample)

print(f"\nObjetivo: {len(target_kepids)} estrellas "
      f"({len(confirmed_sample)} CONFIRMED + {len(sin_koi_sample)} Sin KOI)")


# ============================================================
# 3. Cargar checkpoint existente (resume automático)
# ============================================================
existing_kepids = set()
existing_data   = []
existing_labels = []

if os.path.exists(X_PATH) and os.path.exists(y_PATH) and os.path.exists(kepid_PATH):
    print(f"\n[CARGANDO CHECKPOINT EXISTENTE]")
    X_old = np.load(X_PATH)
    y_old = np.load(y_PATH)
    k_old = np.load(kepid_PATH).tolist()

    existing_data   = list(X_old)
    existing_labels = list(y_old)
    existing_kepids = set(k_old)

    print(f"  Encontradas {len(existing_kepids)} estrellas ya procesadas")
else:
    print("\n[INICIANDO DESDE CERO]")


# ============================================================
# 4. Determinar las estrellas faltantes
# ============================================================
pending = [
    (k, l) for k, l in zip(target_kepids, target_labels)
    if k not in existing_kepids
]
print(f"\nFaltan por descargar: {len(pending)}")


# ============================================================
# 5. Descargar y procesar las faltantes
# ============================================================

def _save_checkpoint(data, labels, kepids):
    """Guarda arrays en disco para permitir reanudación."""
    X_arr = np.array(data, dtype=np.float32)
    y_arr = np.array(labels, dtype=np.int64)
    k_arr = np.array(kepids, dtype=np.int64)

    np.save(X_PATH, X_arr)
    np.save(y_PATH, y_arr)
    np.save(kepid_PATH, k_arr)


data_list   = list(existing_data)
labels_list = list(existing_labels)
kepids_list = list(existing_kepids)

t_start = time.time()

for i, (kepid, label) in enumerate(pending):
    i_global = len(data_list) + 1
    elapsed = time.time() - t_start

    # Progreso cada 10 estrellas
    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i_global}/{len(target_kepids)}] "
              f"KIC {kepid} (label={label}) | {elapsed:.0f}s", flush=True)

    try:
        # 1. Descargar curva real
        lc = get_stitched_lightcurve(int(kepid))
        if lc is None:
            continue

        # 2. Preprocesamiento preservando tránsitos
        flux = preprocess_lightcurve_own_v2(lc)
        if flux is None:
            continue

        # 3. Alineación a TARGET_LENGTH_DS
        flux_aligned = align_to_length(flux, TARGET_LENGTH_DS)
        if flux_aligned is None:
            continue

        # 4. Guardar en memoria
        data_list.append(flux_aligned)
        labels_list.append(label)
        kepids_list.append(kepid)

        # 5. Checkpoint periódico
        if len(data_list) % CHECKPOINT_EVERY == 0:
            _save_checkpoint(data_list, labels_list, kepids_list)
            print(f"    [CHECKPOINT] {len(data_list)} estrellas guardadas", flush=True)

    except Exception as e:
        print(f"    [WARN] KIC {kepid}: {type(e).__name__}: {e}")
        continue


# ============================================================
# 6. Guardado final del dataset v5
# ============================================================
_save_checkpoint(data_list, labels_list, kepids_list)

X_final = np.load(X_PATH)
y_final = np.load(y_PATH)
k_final = np.load(kepid_PATH)

print(f"\n{'='*60}")
print(f"Dataset {OUTPUT_PREFIX} guardado en {DATA_DIR}/")
print(f"  {X_PATH}     shape={X_final.shape}")
print(f"  {y_PATH}     shape={y_final.shape}")
print(f"  {kepid_PATH} shape={k_final.shape}")
print(f"Distribucion: {np.bincount(y_final)}")
print(f"Tiempo total: {time.time()-t_start:.0f}s")
print(f"{'='*60}")


Obteniendo listas de kepids...
CONFIRMED disponibles: 1975
Sin KOI disponibles: 191824

Objetivo: 2800 estrellas (1400 CONFIRMED + 1400 Sin KOI)

[CARGANDO CHECKPOINT EXISTENTE]
  Encontradas 2000 estrellas ya procesadas

Faltan por descargar: 800
  [2001/2800] KIC 5025887 (label=0) | 0s
  [2010/2800] KIC 4366111 (label=0) | 67s
  [2020/2800] KIC 2696938 (label=0) | 118s
  [2030/2800] KIC 8571311 (label=0) | 261s
  [2040/2800] KIC 8309175 (label=0) | 447s
  [2050/2800] KIC 10255036 (label=0) | 668s
    [CHECKPOINT] 2050 estrellas guardadas
  [2060/2800] KIC 1431381 (label=0) | 893s
  [2070/2800] KIC 8671949 (label=0) | 1112s
  [2080/2800] KIC 5878396 (label=0) | 1343s
  [2090/2800] KIC 9632432 (label=0) | 1578s
  [2100/2800] KIC 7868689 (label=0) | 1808s
    [CHECKPOINT] 2100 estrellas guardadas
  [2110/2800] KIC 5706288 (label=0) | 2001s
  [2120/2800] KIC 4647681 (label=0) | 2197s
  [2130/2800] KIC 4730648 (label=0) | 2423s
  [2140/2800] KIC 8073672 (label=0) | 2634s
  [2150/2800] KIC

In [6]:
### --------------------------------------------
### Verificación
### --------------------------------------------

X_v5 = np.load(f'{DATA_DIR}/v5_X.npy')
y_v5 = np.load(f'{DATA_DIR}/v5_y.npy')
k_v5 = np.load(f'{DATA_DIR}/v5_kepids.npy')

print(f"X: {X_v5.shape}")
print(f"y: {y_v5.shape}   clases: {np.bincount(y_v5)}")
print(f"Kepids: {k_v5.shape}")
print(f"Duplicados: {len(k_v5) - len(set(k_v5))}")


X: (2800, 1001)
y: (2800,)   clases: [1400 1400]
Kepids: (2800,)
Duplicados: 0


In [7]:
# ============================================================
# Carga dataset v5 (1200 + 1200), split y preprocessing
# ============================================================
# X_v5: curvas reales interpoladas a 1001 puntos
# y_v5: etiquetas (1 = CONFIRMED, 0 = Sin KOI)
#
# Este experimento es el DEFINITIVO porque:
#   ✔ 2400 estrellas → validación con ~384 muestras → AUC estable (±0.02)
#   ✔ Balance perfecto 1200/1200 → NO requiere sampler
#   ✔ Mismo pipeline v2 (preprocesamiento que preserva tránsitos)
#   ✔ TARGET_LENGTH = 1001 → modelo compacto y rápido
#
# Este dataset v5 es el más robusto y estadísticamente estable
# que has construido hasta ahora.
# ============================================================

# ------------------------------------------------------------
# 1. Cargar dataset v5 desde disco
# ------------------------------------------------------------
X_v5 = np.load(f'{DATA_DIR}/v5_X.npy')
y_v5 = np.load(f'{DATA_DIR}/v5_y.npy')

print(f"X_v5: {X_v5.shape}  y_v5: {y_v5.shape}")
print(f"Distribucion: {np.bincount(y_v5)}")

# ------------------------------------------------------------
# 2. Split Train/Test (estratificado)
# ------------------------------------------------------------
# Se mantiene la proporción 1200/1200 en ambos splits.
# test_size=0.2 → 480 muestras en test.
# Esto permite evaluar el modelo con suficiente estabilidad.
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_v5, y_v5,
    test_size=0.2,
    stratify=y_v5,
    random_state=42
)

print(f"\nTrain: {X_train.shape}  clases: {np.bincount(y_train)}")
print(f"Test:  {X_test.shape}   clases: {np.bincount(y_test)}")

# ------------------------------------------------------------
# 3. Normalización robusta (MAD scaling)
# ------------------------------------------------------------
# Este preprocesamiento:
#   ✔ Centra por mediana
#   ✔ Escala por MAD (Median Absolute Deviation)
#   ✔ Es robusto a outliers instrumentales
#   ✔ Preserva tránsitos planetarios
#
# Es el mismo método usado en v4‑3197 y v4b‑1001 → comparabilidad total.
# ------------------------------------------------------------

def robust_normalize_simple(X, mad_floor=0.05):
    median = np.median(X, axis=1, keepdims=True)
    mad    = np.median(np.abs(X - median), axis=1, keepdims=True)
    mad    = np.maximum(mad, mad_floor * X.std(axis=1, keepdims=True))
    return (X - median) / (1.4826 * mad + 1e-8)

X_train_p = np.clip(robust_normalize_simple(X_train), -15, 15)
X_test_p  = np.clip(robust_normalize_simple(X_test),  -15, 15)

print(f"\nTrain normalizado:  media={X_train_p.mean():.3f}  std={X_train_p.std():.3f}")
print(f"Test normalizado:   media={X_test_p.mean():.3f}  std={X_test_p.std():.3f}")

# ------------------------------------------------------------
# 4. Split interno Train/Val (sin augmentación)
# ------------------------------------------------------------
# 80/20 estratificado:
#   - Train: ~1536 muestras
#   - Val:   ~384 muestras
#
# Con 384 muestras en validación, las métricas (F1, AUC) son
# estadísticamente estables y no dependen del azar.
# ------------------------------------------------------------

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_p, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

print(f"\nTrain: {X_tr.shape}  clases: {np.bincount(y_tr)}")
print(f"Val:   {X_val.shape}  clases: {np.bincount(y_val)}")

# ------------------------------------------------------------
# 5. DataLoaders (sin augmentación, sin sampler)
# ------------------------------------------------------------
# Como el dataset está perfectamente balanceado (1200/1200),
# NO se necesita WeightedRandomSampler.
#
# Esto simplifica el pipeline y acelera el entrenamiento.
# ------------------------------------------------------------

def make_loaders(X_tr, y_tr, X_val, y_val, batch_size=32):
    train_ds = TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr))
    val_ds   = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

train_loader, val_loader = make_loaders(X_tr, y_tr, X_val, y_val)
print(f"\nBatches train: {len(train_loader)}  val: {len(val_loader)}")


X_v5: (2800, 1001)  y_v5: (2800,)
Distribucion: [1400 1400]

Train: (2240, 1001)  clases: [1120 1120]
Test:  (560, 1001)   clases: [280 280]

Train normalizado:  media=-0.022  std=1.197
Test normalizado:   media=-0.021  std=1.167

Train: (1792, 1001)  clases: [896 896]
Val:   (448, 1001)  clases: [224 224]

Batches train: 56  val: 14


In [8]:
# ============================================================
# Arquitectura FlyInspiredSNN (versión adaptable a TARGET_LENGTH)
# ============================================================
# Esta arquitectura está inspirada en el sistema visual de la mosca:
#
#   - Extractor jerárquico (análogos a lámina → médula)
#       conv1 → pool1 → conv2 → pool2
#
#   - Proyección densa + SNN (análogos a lóbulas)
#       fc_in → snn_fc → LIF → fc_out
#
# Características clave:
#   ✔ input_len se adapta automáticamente a TARGET_LENGTH
#   ✔ flat_dim se calcula dinámicamente con un dummy
#   ✔ num_steps controla el unfolding temporal de la SNN
#   ✔ LIF (Leaky Integrate-and-Fire) con reset zero
#
# Esta versión funciona tanto para:
#   - v4‑3197 (curvas largas)
#   - v4b‑1001 (curvas compactas)
# ============================================================

class FlyInspiredSNN(nn.Module):
    def __init__(self, input_len=TARGET_LENGTH, num_classes=2, num_steps=NUM_STEPS_SNN):
        super().__init__()
        self.num_steps = num_steps

        # ------------------------------
        # Extractor jerárquico
        # ------------------------------
        self.conv1 = nn.Conv1d(1, 8, kernel_size=15, stride=3, padding=7)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(8, 16, kernel_size=9, stride=3, padding=4)
        self.pool2 = nn.MaxPool1d(2)

        self.relu  = nn.ReLU()
        self.dropout = nn.Dropout(0.4)

        # ------------------------------
        # Cálculo dinámico del tamaño plano
        # ------------------------------
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_len)
            dummy = self.pool1(self.relu(self.conv1(dummy)))
            dummy = self.pool2(self.relu(self.conv2(dummy)))
            self.flat_dim = dummy.numel()

        # ------------------------------
        # Capa densa + SNN
        # ------------------------------
        self.fc_in  = nn.Linear(self.flat_dim, 32)
        self.snn_fc = nn.Linear(32, 16)

        self.lif    = snn.Leaky(beta=0.9, threshold=1.0, reset_mechanism="zero")
        self.fc_out = nn.Linear(16, num_classes)

    def forward(self, x):
        # x: (batch, 1001) → agregar canal
        x = x.unsqueeze(1)

        # Extractor jerárquico
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))

        # Aplanar
        x = x.flatten(start_dim=1)

        # Proyección densa
        x = self.dropout(self.relu(self.fc_in(x)))

        # SNN temporal
        mem = self.lif.init_leaky()
        spike_accum = torch.zeros(x.size(0), 2, device=x.device)

        for _ in range(self.num_steps):
            cur = self.snn_fc(x)
            spk, mem = self.lif(cur, mem)
            spike_accum += self.fc_out(spk)

        return spike_accum / self.num_steps


# ============================================================
# Función de entrenamiento para FlyInspiredSNN
# ============================================================
# Características:
#   ✔ CrossEntropyLoss
#   ✔ Adam + weight_decay
#   ✔ ReduceLROnPlateau (paciencia=7)
#   ✔ Clip de gradiente (1.0)
#   ✔ Guarda el mejor modelo según F1 en validación
#   ✔ Calcula AUC y F1 por época
#
# Compatible con:
#   - v4‑3197
#   - v4b‑1001
# ============================================================

def train_model(model, train_loader, val_loader, epochs=30, lr=5e-4):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=7, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}
    best_f1 = 0.0
    best_state = None
    
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        train_loss = 0.0

        # ------------------------------
        # Entrenamiento
        # ------------------------------
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()

            out = model(xb)
            loss = criterion(out, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # ------------------------------
        # Validación
        # ------------------------------
        model.eval()
        val_loss = 0.0
        all_probs, all_preds, all_labels = [], [], []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)

                val_loss += criterion(out, yb).item()
                all_probs.append(torch.softmax(out, dim=1)[:,1].cpu().numpy())
                all_preds.append(out.argmax(dim=1).cpu().numpy())
                all_labels.append(yb.cpu().numpy())

        val_loss /= len(val_loader)

        all_probs  = np.concatenate(all_probs)
        all_preds  = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)

        val_f1  = f1_score(all_labels, all_preds, zero_division=0)
        val_auc = roc_auc_score(all_labels, all_probs) if len(np.unique(all_labels)) > 1 else 0.0

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)

        # ------------------------------
        # Guardar mejor modelo
        # ------------------------------
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            torch.save(
                {'model_state': best_state, 'f1': best_f1, 'auc': val_auc},
                f'{MODELS_DIR}/snn_v4b_best.pt'
            )

        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']

        print(f"Epoca {epoch+1:>2}/{epochs} | "
              f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"F1: {val_f1:.4f} | AUC: {val_auc:.4f} | {elapsed:.1f}s", flush=True)
    
    # Cargar mejor estado
    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"\n[OK] Mejor F1 val: {best_f1:.4f}")
    return history


print("Arquitectura y train_model definidas.")


Arquitectura y train_model definidas.


In [9]:
# ============================================================
# Instanciar modelo FlyInspiredSNN para TARGET_LENGTH = 1001
# ============================================================
# Esta instancia usa:
#   - input_len = 1001  (nuevo tamaño compacto)
#   - num_steps = 25    (unfolding temporal de la SNN)
#   - Arquitectura idéntica a v4‑3197, pero recalculando flat_dim
#
# ¿Por qué es importante?
#   ✔ Permite comparar directamente modelos entrenados con 3197 vs 1001 puntos
#   ✔ flat_dim cambia automáticamente según TARGET_LENGTH
#   ✔ Se mantiene la misma estructura bioinspirada
# ============================================================

model_1001 = FlyInspiredSNN(input_len=TARGET_LENGTH).to(DEVICE)

# Número total de parámetros
print(f"Params: {sum(p.numel() for p in model_1001.parameters()):,}")

# Tamaño del vector plano después de conv/pool
print(f"flat_dim: {model_1001.flat_dim}")

# ============================================================
# Entrenamiento del modelo FlyInspiredSNN‑1001
# ============================================================
# Se usa exactamente el mismo pipeline de entrenamiento que en v4‑3197:
#
#   ✔ CrossEntropyLoss
#   ✔ Adam (lr=5e-4, weight_decay=1e-4)
#   ✔ ReduceLROnPlateau (paciencia=7)
#   ✔ Clip de gradiente (1.0)
#   ✔ F1 y AUC por época
#   ✔ Guardado del mejor modelo según F1 en validación
#
# Esto garantiza comparabilidad científica entre:
#   - Modelo v4‑3197
#   - Modelo v4b‑1001
# ============================================================

print("\nIniciando entrenamiento con TARGET_LENGTH=1001...", flush=True)
history_1001 = train_model(
    model_1001,
    train_loader,
    val_loader,
    epochs=30,
    lr=5e-4
)


Params: 16,226
flat_dim: 448

Iniciando entrenamiento con TARGET_LENGTH=1001...
Epoca  1/30 | Train: 0.6952 | Val: 0.6927 | F1: 0.6514 | AUC: 0.5383 | 4.4s
Epoca  2/30 | Train: 0.6894 | Val: 0.6873 | F1: 0.6451 | AUC: 0.6099 | 1.5s
Epoca  3/30 | Train: 0.6679 | Val: 0.6739 | F1: 0.6066 | AUC: 0.6144 | 1.5s
Epoca  4/30 | Train: 0.6324 | Val: 0.6712 | F1: 0.5977 | AUC: 0.6283 | 1.4s
Epoca  5/30 | Train: 0.6211 | Val: 0.6717 | F1: 0.6214 | AUC: 0.6370 | 1.4s
Epoca  6/30 | Train: 0.6063 | Val: 0.6726 | F1: 0.5542 | AUC: 0.6467 | 1.7s
Epoca  7/30 | Train: 0.6024 | Val: 0.6718 | F1: 0.5885 | AUC: 0.6469 | 1.8s
Epoca  8/30 | Train: 0.5936 | Val: 0.6681 | F1: 0.5821 | AUC: 0.6541 | 1.6s
Epoca  9/30 | Train: 0.5855 | Val: 0.6650 | F1: 0.6424 | AUC: 0.6619 | 1.6s
Epoca 10/30 | Train: 0.5734 | Val: 0.6625 | F1: 0.6263 | AUC: 0.6621 | 1.6s
Epoca 11/30 | Train: 0.5680 | Val: 0.6755 | F1: 0.5474 | AUC: 0.6597 | 1.7s
Epoca 12/30 | Train: 0.5583 | Val: 0.6663 | F1: 0.6130 | AUC: 0.6588 | 1.8s
Epoca 13

In [10]:
# ============================================================
# Función: find_best_threshold
# ============================================================
# Objetivo:
#   Encontrar el umbral óptimo para convertir probabilidades
#   en predicciones binarias (0/1) maximizando el F1-score.
#
# Flujo:
#   1. Obtener probabilidades del modelo sobre el conjunto de validación.
#   2. Barrer umbrales desde 0.05 hasta 0.95 en pasos de 0.02.
#   3. Calcular F1 para cada umbral.
#   4. Seleccionar el umbral con mayor F1.
#
# Razón:
#   - En datasets desbalanceados, el umbral 0.50 NO es óptimo.
#   - El F1 es más estable que accuracy cuando la clase positiva
#     es minoritaria (CONFIRMED).
# ============================================================

def find_best_threshold(model, val_loader):
    model.eval()

    all_probs, all_labels = [], []

    # ------------------------------
    # Obtener probabilidades reales
    # ------------------------------
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            probs = torch.softmax(model(xb), dim=1)[:, 1]
            all_probs.append(probs.cpu().numpy())
            all_labels.append(yb.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    # ------------------------------
    # Barrido de umbrales
    # ------------------------------
    thresholds = np.arange(0.05, 0.95, 0.02)
    f1s = [
        f1_score(all_labels, (all_probs > t).astype(int), zero_division=0)
        for t in thresholds
    ]

    best_idx = np.argmax(f1s)

    print(f"Mejor umbral: {thresholds[best_idx]:.2f}")
    print(f"Mejor F1 val: {f1s[best_idx]:.4f}")

    return thresholds[best_idx]


# ============================================================
# Calcular umbral óptimo para modelo_1001
# ============================================================

best_thr = find_best_threshold(model_1001, val_loader)

# ============================================================
# Evaluación en TEST con el umbral óptimo
# ============================================================
# Flujo:
#   1. Crear DataLoader de test
#   2. Obtener probabilidades del modelo
#   3. Convertir a predicciones usando best_thr
#   4. Calcular:
#       ✔ classification_report
#       ✔ AUC-ROC
#       ✔ Matriz de confusión
#
# Esto permite comparar directamente:
#   - Modelo v4‑3197
#   - Modelo v4b‑1001
# ============================================================

test_ds = TensorDataset(torch.FloatTensor(X_test_p), torch.LongTensor(y_test))
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

model_1001.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        probs = torch.softmax(model_1001(xb), dim=1)[:, 1]
        all_probs.append(probs.cpu().numpy())
        all_labels.append(yb.cpu().numpy())

all_probs  = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_preds  = (all_probs > best_thr).astype(int)

# ============================================================
# Reporte final de TEST
# ============================================================

print(f"\n=== TEST con umbral {best_thr:.2f} ===")
print(classification_report(
    all_labels,
    all_preds,
    target_names=['Sin senal', 'Planeta'],
    digits=4,
    zero_division=0
))

print(f"AUC-ROC: {roc_auc_score(all_labels, all_probs):.4f}")

cm = confusion_matrix(all_labels, all_preds)
print(f"\nMatriz: TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")


Mejor umbral: 0.05
Mejor F1 val: 0.6667

=== TEST con umbral 0.05 ===
              precision    recall  f1-score   support

   Sin senal     0.0000    0.0000    0.0000       280
     Planeta     0.5000    1.0000    0.6667       280

    accuracy                         0.5000       560
   macro avg     0.2500    0.5000    0.3333       560
weighted avg     0.2500    0.5000    0.3333       560

AUC-ROC: 0.5588

Matriz: TN=0  FP=280  FN=0  TP=280


In [12]:
import json
import os

os.makedirs(f'{OUTPUTS_DIR}', exist_ok=True)

resultados = {
    'dataset': 'kepler_v5',
    'n_total': 2800,
    'n_per_class': 1400,
    'target_length': 1001,
    'train': 2240,
    'test': 560,
    'val': 448,
    'arquitectura': 'FlyInspiredSNN',
    'params': 16226,
    'epochs': 30,
    'lr': 5e-4,
    'resultados_test': {
        'auc_roc': 0.5588,
        'f1_planeta': 0.6667,
        'threshold': 0.05,
        'tn': 0, 'fp': 280, 'fn': 0, 'tp': 280,
    },
    'resultados_val': {
        'mejor_f1': 0.6514,
        'mejor_auc': 0.6621,
    },
    'conclusion': (
        'La SNN no distingue CONFIRMED de Sin KOI usando solo curvas de luz '
        'downsampleadas a 1001 puntos. El AUC de 0.56 está cerca del azar. '
        'El modelo colapsa a predecir "planeta" para todas las muestras '
        'de test (TN=0, FP=280). La tarea requiere features adicionales '
        '(periodo, duracion, profundidad, centroide, espectroscopia) que son '
        'estandar en la literatura de deteccion de exoplanetas.'
    ),
}

with open(f'{OUTPUTS_DIR}/v5_resultados.json', 'w') as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

print("[OK] Resultados guardados en", f'{OUTPUTS_DIR}/v5_resultados.json')
print()
for k, v in resultados.items():
    if k != 'conclusion':
        print(f"  {k}: {v}")
print(f"\n  conclusion: {resultados['conclusion'][:100]}...")

[OK] Resultados guardados en ../outputs/v5_resultados.json

  dataset: kepler_v5
  n_total: 2800
  n_per_class: 1400
  target_length: 1001
  train: 2240
  test: 560
  val: 448
  arquitectura: FlyInspiredSNN
  params: 16226
  epochs: 30
  lr: 0.0005
  resultados_test: {'auc_roc': 0.5588, 'f1_planeta': 0.6667, 'threshold': 0.05, 'tn': 0, 'fp': 280, 'fn': 0, 'tp': 280}
  resultados_val: {'mejor_f1': 0.6514, 'mejor_auc': 0.6621}

  conclusion: La SNN no distingue CONFIRMED de Sin KOI usando solo curvas de luz downsampleadas a 1001 puntos. El ...
